# What makes the toxic reviews toxic? Evidence phrases and types

For every review that is still toxic after the cheating re-check (1,669), Gemini points at up to 3 **exact words or phrases** that make it toxic and gives each one a **type**. Then we count.

**Types** (shown with plain names in the tables) follow common toxicity schemes: Perspective API attributes (insult, profanity, identity attack, threat, sexually explicit), Risch and Krestel (2020: profanity, insults, threats, identity hate, otherwise toxic), and the game-toxicity literature (griefing, trolling, toxic community; Beres et al. 2021; Fair Play Alliance and ADL *Disruption and Harms in Online Gaming Framework*, 2020). For identity attacks Gemini also says which group (ethnicity or nationality, gender, and so on). Each piece of evidence also has a **target**: teammate, opponent, players in general, a nationality, the developer.

**Safety check.** A quote must appear word for word in the review. `quote_in_review` is `False` when it does not, so invented quotes are caught and can be dropped.

**Caveats.** The evidence is Gemini's own explanation, not proof of why it decided. Counts describe Gemini's labels, not player behavior. A phrase that points at a nationality is a statement in the review, not a fact about that nationality.

Reads `data/gemini_harm_recheck_*.csv` (read only) and writes NEW files: `data/gemini_toxic_evidence_*.jsonl` and `.csv`.

**Run All twice:** first with `SAMPLE = 50`, then `SAMPLE = None` for all.

## 1. Setup

In [ ]:
import os, sys, json, importlib
import pandas as pd
from dotenv import load_dotenv
from google import genai

sys.path.insert(0, "../scripts")
import toxic_evidence
importlib.reload(toxic_evidence)

load_dotenv("../.env")
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 2. Choose the reviews (the one thing to change)

In [ ]:
SAMPLE = 50   # <-- test first with 50 toxic reviews. Then set to None and Run All again.

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0
MODEL_TAG = GEMINI_MODEL.replace("/", "_")

SOURCE = f"../data/gemini_harm_recheck_{MODEL_TAG}.csv"     # read only
CACHE  = f"../data/gemini_toxic_evidence_{MODEL_TAG}.jsonl" # NEW
OUT    = f"../data/gemini_toxic_evidence_{MODEL_TAG}.csv"   # NEW

rc = pd.read_csv(SOURCE)
toxic = rc[rc["harm_mention_v2"] == "yes"].sample(frac=1, random_state=0).reset_index(drop=True)  # fixed shuffle
df = toxic.head(SAMPLE) if SAMPLE else toxic
print(f"{len(toxic):,} toxic reviews in total, this run covers {len(df):,}")
df["game"].value_counts()

## 3. Run

Every review is saved the moment it finishes. A second run skips what is saved.

In [ ]:
with open("../prompts/toxic_evidence.txt", "r") as f:
    SYSTEM_EV = f.read()

run = toxic_evidence.EvidenceRun(df, client, GEMINI_MODEL, SYSTEM_EV, CACHE, temperature=TEMPERATURE, workers=8)
run.start()

## 4. Save the evidence table (new file) and check the quotes

In [ ]:
ev = run.export(OUT)

ok_rate = ev["quote_in_review"].mean()
print(f"\nQuotes found word for word in the review: {ev['quote_in_review'].sum():,} of {len(ev):,} ({ok_rate:.1%})")
print("The rest are kept in the file but left out of the counts below.")
ev = ev[ev["quote_in_review"]].copy()
ev.head(10)[["game", "quote", "meaning_en", "type", "identity_group", "target"]]

## 5. Summary

First a short written summary, then one table: for each type of toxicity, how many toxic reviews have it, the share per game, the most common phrases, and a couple of readable examples.

In [ ]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 250)

L = toxic_evidence
ev["Type"] = ev["type"].map(L.TYPE_LABELS)
ev["Aimed at"] = ev["target"].map(L.TARGET_LABELS)
ev["Group"] = ev["identity_group"].map(L.GROUP_LABELS)
ev = ev.merge(df[["key", "language"]].rename(columns={"language": "lang2"}), on="key", how="left")
ev["language"] = ev["language"].fillna(ev["lang2"])

n_toxic = df["key"].nunique()
n_with = ev["key"].nunique()
per_type = ev.drop_duplicates(["key", "type"]).groupby("type")["key"].nunique().sort_values(ascending=False)
top = per_type.index[0]
per_target = ev.drop_duplicates(["key", "target"]).groupby("target")["key"].nunique().sort_values(ascending=False)

print(f"{n_toxic:,} toxic reviews were checked. {n_with:,} ({n_with / n_toxic:.0%}) have at least one quoted phrase that was found in the review.")
print(f"Most common type: {L.TYPE_LABELS[top]}, in {per_type.iloc[0] / n_toxic:.0%} of toxic reviews.")
print("Order of types: " + ", ".join(f"{L.TYPE_LABELS[t].split(' (')[0]} {v / n_toxic:.0%}" for t, v in per_type.items()))
print(f"Most often aimed at: {L.TARGET_LABELS[per_target.index[0]]} ({per_target.iloc[0] / n_toxic:.0%} of toxic reviews), then {L.TARGET_LABELS[per_target.index[1]]} ({per_target.iloc[1] / n_toxic:.0%}).")
print("A review can have several types, so the shares add up to more than 100%.")

In [ ]:
rows = []
n_game = df.groupby("game")["key"].nunique()
for t in per_type.index:
    s = ev[ev["type"] == t]
    byg = s.drop_duplicates("key").groupby("game")["key"].nunique()
    rows.append({
        "Type": L.TYPE_LABELS[t],
        "Toxic reviews": s["key"].nunique(),
        "% of toxic reviews": round(s["key"].nunique() / n_toxic * 100, 1),
        "CS2 %": round(byg.get("Counter-Strike 2", 0) / n_game.get("Counter-Strike 2", 1) * 100, 1),
        "GTA V %": round(byg.get("Grand Theft Auto V", 0) / n_game.get("Grand Theft Auto V", 1) * 100, 1),
        "PUBG %": round(byg.get("PUBG: BATTLEGROUNDS", 0) / n_game.get("PUBG: BATTLEGROUNDS", 1) * 100, 1),
        "Common phrases (English meaning)": ", ".join(s["meaning_en"].str.lower().value_counts().head(4).index),
        "Examples": L.examples(s, n=2),
    })
summary = pd.DataFrame(rows)
summary

## 6. Who is it aimed at?

In [ ]:
rows = []
for t in per_target.index:
    s = ev[ev["target"] == t]
    rows.append({"Aimed at": L.TARGET_LABELS[t], "Toxic reviews": s["key"].nunique(),
                 "% of toxic reviews": round(s["key"].nunique() / n_toxic * 100, 1), "Examples": L.examples(s, n=2)})
pd.DataFrame(rows)

## 7. Attacks on identity: which group?

Only the evidence typed as an attack on who someone is.

In [ ]:
ia = ev[ev["type"] == "identity_attack"]
print(f"{ia['key'].nunique():,} toxic reviews ({ia['key'].nunique() / n_toxic:.0%}) have an attack on identity.\n")
rows = []
for g, s in ia.groupby("identity_group"):
    rows.append({"Group": L.GROUP_LABELS[g], "Toxic reviews": s["key"].nunique(),
                 "% of identity attacks": round(s["key"].nunique() / ia["key"].nunique() * 100, 1),
                 "Examples": L.examples(s, n=2)})
pd.DataFrame(rows).sort_values("Toxic reviews", ascending=False).reset_index(drop=True)

## 8. By game

Number of toxic reviews with an attack on identity, per game.

In [ ]:
ia.drop_duplicates("key").groupby("game").size().to_frame("Reviews with an attack on identity").join(n_game.rename("Toxic reviews"))

## 9. Most common phrases per type

The top meanings in English, so phrases in different languages are counted together.

In [ ]:
for t in per_type.index:
    s = ev[ev["type"] == t]
    print(f"\n=== {L.TYPE_LABELS[t]}: {s['key'].nunique():,} reviews ===")
    print(s["meaning_en"].str.lower().value_counts().head(8).to_string())

## 10. Human spot check

Gemini explains its own labels, so check a few by hand. This writes 30 random pieces of evidence to a new CSV. Fill `human_ok` with `yes` or `no` (is the type right, and does the quote really show toxicity?). The share of `yes` tells us how far to trust the counts above.

In [ ]:
check = ev.sample(min(30, len(ev)), random_state=7)[["key", "game", "quote", "meaning_en", "Type", "Group", "Aimed at", "review"]].copy()
check["human_ok"] = ""
CHECK_OUT = f"../data/toxic_evidence_spotcheck_{MODEL_TAG}.csv"
check.to_csv(CHECK_OUT, index=False)
print(f"Saved {len(check)} rows -> {CHECK_OUT}")
check[["quote", "meaning_en", "Type", "Aimed at"]].head(10)